In [1]:
!pip install -q transformers datasets scikit-learn accelerate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 67.8 MB/s eta 0:00:00:00:01:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have n

In [2]:
"""
ClauseOps — Phase 4 v5.0: Train Obligation Models on Kaggle
==============================================================
Fine-tunes TWO Legal-BERT models for 100% offline obligation detection:

  Model A: Modality Classifier (OBLIGATION/PROHIBITION/PERMISSION/DECLARATIVE)
  Model B: Agent + Action NER (BIO token classification)

HOW TO USE ON KAGGLE:
  1. Upload the `training_data/` folder as a Kaggle Dataset
  2. Create a new Kaggle Notebook with GPU (P100 or T4)
  3. Copy each "=== CELL N ===" block into a separate notebook cell
  4. Run cells in order
  5. Cell 7 trains Model A (~30-60 min on P100)
  6. Cell 10 trains Model B (~30-60 min on P100)
  7. Download both models from /kaggle/working/

Training Data Format (generated by generate_training_data.py):
  modality_*.jsonl: {"text": str, "label": int, "label_name": str}
  ner_*.jsonl: {"tokens": list[str], "tags": list[int], "text": str}
"""

# =============================================================================
# === CELL 1: Install Dependencies ===
# =============================================================================

# !pip install -q transformers datasets scikit-learn accelerate seqeval

# =============================================================================
# === CELL 2: Imports & GPU Check ===
# =============================================================================

import os
import json
import time
import numpy as np
import torch
from pathlib import Path
from collections import Counter

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorForTokenClassification,
)
from datasets import Dataset
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight
from torch.nn import CrossEntropyLoss

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU Memory: {gpu_mem:.1f} GB")

# =============================================================================
# === CELL 3: Load Training Data ===
# =============================================================================
# Update DATA_DIR to match your Kaggle dataset path

# On Kaggle, datasets are mounted at /kaggle/input/<dataset-name>/
# If you uploaded the folder as "clauseops-training-data":
DATA_DIR = Path("/kaggle/input/datasets/uday100/clauseops-obligation-training-data/training_data")

# Fallback for local testing
if not DATA_DIR.exists():
    DATA_DIR = Path("./training_data")

if not DATA_DIR.exists():
    raise FileNotFoundError(
        f"Training data not found at {DATA_DIR}.\n"
        "Upload the training_data/ folder as a Kaggle Dataset first."
    )

# Load metadata
with open(DATA_DIR / "metadata.json") as f:
    metadata = json.load(f)

print(f"Metadata: {json.dumps(metadata, indent=2)}")

# Load modality data
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

mod_train = load_jsonl(DATA_DIR / "modality_train.jsonl")
mod_val = load_jsonl(DATA_DIR / "modality_val.jsonl")
mod_test = load_jsonl(DATA_DIR / "modality_test.jsonl")

print(f"\n=== Modality Data ===")
print(f"Train: {len(mod_train)}")
print(f"Val:   {len(mod_val)}")
print(f"Test:  {len(mod_test)}")

# Class distribution
mod_labels = [d["label"] for d in mod_train]
ID_TO_MODALITY = {v: k for k, v in metadata["modality_labels"].items()}
print(f"\nTrain class distribution:")
for label_id, count in sorted(Counter(mod_labels).items()):
    print(f"  {ID_TO_MODALITY[label_id]:15s}: {count}")

# Load NER data
ner_train = load_jsonl(DATA_DIR / "ner_train.jsonl")
ner_val = load_jsonl(DATA_DIR / "ner_val.jsonl")
ner_test = load_jsonl(DATA_DIR / "ner_test.jsonl")

print(f"\n=== NER Data ===")
print(f"Train: {len(ner_train)}")
print(f"Val:   {len(ner_val)}")
print(f"Test:  {len(ner_test)}")


# =============================================================================
# === CELL 4: Load Legal-BERT Tokenizer ===
# =============================================================================

MODEL_NAME = "nlpaueb/legal-bert-base-uncased"
MAX_LENGTH = 512

print(f"Loading tokenizer: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Vocab size: {tokenizer.vocab_size}")


# =============================================================================
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL A: MODALITY CLASSIFIER
# ═══════════════════════════════════════════════════════════════════════════════
# =============================================================================


# =============================================================================
# === CELL 5: Prepare Modality Dataset ===
# =============================================================================

NUM_MODALITY_LABELS = len(metadata["modality_labels"])
print(f"Number of modality classes: {NUM_MODALITY_LABELS}")

# Create HuggingFace datasets
mod_train_ds = Dataset.from_dict({
    "text": [d["text"] for d in mod_train],
    "label": [d["label"] for d in mod_train],
})
mod_val_ds = Dataset.from_dict({
    "text": [d["text"] for d in mod_val],
    "label": [d["label"] for d in mod_val],
})
mod_test_ds = Dataset.from_dict({
    "text": [d["text"] for d in mod_test],
    "label": [d["label"] for d in mod_test],
})

# Tokenize
def tokenize_modality(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding="max_length",
    )

mod_train_tok = mod_train_ds.map(tokenize_modality, batched=True, batch_size=256,
                                  remove_columns=["text"])
mod_val_tok = mod_val_ds.map(tokenize_modality, batched=True, batch_size=256,
                              remove_columns=["text"])
mod_test_tok = mod_test_ds.map(tokenize_modality, batched=True, batch_size=256,
                                remove_columns=["text"])

mod_train_tok.set_format("torch")
mod_val_tok.set_format("torch")
mod_test_tok.set_format("torch")

print(f"Tokenized modality datasets ready.")
print(f"  Train: {len(mod_train_tok)}")


# =============================================================================
# === CELL 6: Modality Classifier Setup ===
# =============================================================================

# Compute class weights for imbalanced data
train_labels_array = np.array([d["label"] for d in mod_train])
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(NUM_MODALITY_LABELS),
    y=train_labels_array,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  {ID_TO_MODALITY[i]:15s}: {w:.3f}")


class WeightedLossTrainer(Trainer):
    """Trainer with class-weighted loss for imbalanced data."""
    def __init__(self, class_weights: torch.Tensor, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self._class_weights = class_weights

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        weights = self._class_weights.to(logits.device)
        loss = CrossEntropyLoss(weight=weights)(logits, labels)
        return (loss, outputs) if return_outputs else loss


def compute_modality_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "accuracy": (preds == labels).mean(),
    }


# Load model
print(f"Loading {MODEL_NAME} for sequence classification...")
mod_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_MODALITY_LABELS,
    ignore_mismatched_sizes=True,
)

total_params = sum(p.numel() for p in mod_model.parameters())
print(f"Total parameters: {total_params:,}")


# =============================================================================
# === CELL 7: TRAIN MODEL A (Modality Classifier) ===
# =============================================================================
# Expected time: ~30-60 min on P100, ~60-90 min on T4

MOD_OUTPUT_DIR = "/kaggle/working/clauseops-modality-classifier"
if not os.path.exists("/kaggle/working"):
    MOD_OUTPUT_DIR = "./clauseops-modality-classifier"

mod_training_args = TrainingArguments(
    output_dir=MOD_OUTPUT_DIR,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=5,
    logging_first_step=True,
    report_to="none",
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

mod_trainer = WeightedLossTrainer(
    class_weights=class_weights_tensor,
    model=mod_model,
    args=mod_training_args,
    train_dataset=mod_train_tok,
    eval_dataset=mod_val_tok,
    compute_metrics=compute_modality_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

print("🚀 Training Model A: Modality Classifier...")
print(f"   Train size: {len(mod_train_tok)}")
print(f"   Epochs: {mod_training_args.num_train_epochs}")
print()

mod_train_result = mod_trainer.train()

print("\n" + "=" * 50)
print("MODEL A TRAINING COMPLETE")
print("=" * 50)
metrics = mod_train_result.metrics
print(f"Training loss: {metrics.get('train_loss', 'N/A'):.4f}")
print(f"Training time: {metrics.get('train_runtime', 0) / 60:.1f} minutes")


# =============================================================================
# === CELL 8: Evaluate Model A ===
# =============================================================================

print("Evaluating Model A on test set...")
mod_eval = mod_trainer.evaluate(mod_test_tok)

print(f"\n{'=' * 50}")
print("MODEL A — TEST SET RESULTS")
print(f"{'=' * 50}")
print(f"Macro-F1:  {mod_eval['eval_macro_f1']:.4f}")
print(f"Micro-F1:  {mod_eval['eval_micro_f1']:.4f}")
print(f"Accuracy:  {mod_eval['eval_accuracy']:.4f}")

# Detailed report
predictions = mod_trainer.predict(mod_test_tok)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids

modality_names = [ID_TO_MODALITY[i] for i in range(NUM_MODALITY_LABELS)]
print(f"\n{'=' * 60}")
print("DETAILED CLASSIFICATION REPORT")
print(f"{'=' * 60}")
print(classification_report(labels, preds, target_names=modality_names, digits=3))


# =============================================================================
# === CELL 9: Save Model A ===
# =============================================================================

mod_save_path = os.path.join(MOD_OUTPUT_DIR, "final")
print(f"Saving Model A to: {mod_save_path}")

mod_trainer.save_model(mod_save_path)
tokenizer.save_pretrained(mod_save_path)

mod_metadata = {
    "model_name": MODEL_NAME,
    "task": "modality_classification",
    "num_labels": NUM_MODALITY_LABELS,
    "label_map": metadata["modality_labels"],
    "id_to_label": {str(v): k for k, v in metadata["modality_labels"].items()},
    "eval_macro_f1": mod_eval["eval_macro_f1"],
    "eval_accuracy": mod_eval["eval_accuracy"],
    "training_date": time.strftime("%Y-%m-%d %H:%M"),
}

with open(os.path.join(mod_save_path, "obligation_metadata.json"), "w") as f:
    json.dump(mod_metadata, f, indent=2)

print("✅ Model A saved!")
for fname in sorted(os.listdir(mod_save_path)):
    fsize = os.path.getsize(os.path.join(mod_save_path, fname))
    print(f"   {fname}: {fsize / 1e6:.1f} MB")


# =============================================================================
# ═══════════════════════════════════════════════════════════════════════════════
# MODEL B: AGENT + ACTION NER
# ═══════════════════════════════════════════════════════════════════════════════
# =============================================================================


PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4
GPU Memory: 15.6 GB
Metadata: {
  "modality_labels": {
    "OBLIGATION": 0,
    "PROHIBITION": 1,
    "PERMISSION": 2,
    "DECLARATIVE": 3
  },
  "ner_tags": {
    "O": 0,
    "B-AGENT": 1,
    "I-AGENT": 2,
    "B-ACTION": 3,
    "I-ACTION": 4
  },
  "total_annotations": 1998,
  "train_size": 1498,
  "val_size": 249,
  "test_size": 251
}

=== Modality Data ===
Train: 1498
Val:   249
Test:  251

Train class distribution:
  OBLIGATION     : 338
  PROHIBITION    : 193
  PERMISSION     : 218
  DECLARATIVE    : 749



=== NER Data ===
Train: 1498
Val:   249
Test:  251
Loading tokenizer: nlpaueb/legal-bert-base-uncased


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La


MODEL A TRAINING COMPLETE
Training loss: 3.4944
Training time: 13.7 minutes
Evaluating Model A on test set...



MODEL A — TEST SET RESULTS
Macro-F1:  0.8632
Micro-F1:  0.8685
Accuracy:  0.8685

DETAILED CLASSIFICATION REPORT
              precision    recall  f1-score   support

  OBLIGATION      0.880     0.772     0.822        57
 PROHIBITION      0.825     0.971     0.892        34
  PERMISSION      0.765     0.951     0.848        41
 DECLARATIVE      0.927     0.857     0.891       119

    accuracy                          0.869       251
   macro avg      0.849     0.888     0.863       251
weighted avg      0.876     0.869     0.868       251

Saving Model A to: /kaggle/working/clauseops-modality-classifier/final


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model A saved!
   config.json: 0.0 MB
   model.safetensors: 438.0 MB
   obligation_metadata.json: 0.0 MB
   tokenizer.json: 0.7 MB
   tokenizer_config.json: 0.0 MB
   training_args.bin: 0.0 MB


In [ ]:

# =============================================================================
# === CELL 10: Prepare NER Dataset ===
# =============================================================================
# The NER data may not have BIO tags pre-computed (if tokenizer wasn't
# available during data generation). In that case, we compute them here.

NER_TAGS = metadata["ner_tags"]  # {"O": 0, "B-AGENT": 1, "I-AGENT": 2, "B-ACTION": 3, "I-ACTION": 4}
NUM_NER_TAGS = len(NER_TAGS)
ID_TO_NER_TAG = {v: k for k, v in NER_TAGS.items()}

print(f"NER Tags: {NER_TAGS}")
print(f"Number of NER tags: {NUM_NER_TAGS}")


def prepare_ner_from_text(examples_list):
    """
    If NER data has raw text + agent/action strings,
    tokenize and create BIO tags from scratch.
    """
    all_input_ids = []
    all_attention_masks = []
    all_labels = []

    for ex in examples_list:
        text = ex.get("text", "")
        agent = ex.get("agent")
        action = ex.get("action")

        # Tokenize with offset mapping
        encoding = tokenizer(
            text,
            truncation=True,
            max_length=MAX_LENGTH,
            padding="max_length",
            return_offsets_mapping=True,
        )

        input_ids = encoding["input_ids"]
        attention_mask = encoding["attention_mask"]
        offsets = encoding["offset_mapping"]

        # Initialize labels (all O, with -100 for special tokens)
        labels = [-100 if att == 0 else 0 for att in attention_mask]

        # Also set -100 for [CLS] and [SEP]
        labels[0] = -100  # [CLS]
        # Find [SEP]
        for idx in range(len(input_ids) - 1, -1, -1):
            if input_ids[idx] == tokenizer.sep_token_id:
                labels[idx] = -100
                break

        bio_tags = ex.get("bio_tags")

        # Try using pre-computed BIO tags first
        if bio_tags and len(bio_tags) > 0:
            for i, tag_dict in enumerate(bio_tags):
                if i >= len(labels):
                    break
                tag = tag_dict["tag"]
                # Only apply if it's not a special token
                if labels[i] != -100 and tag in NER_TAGS:
                    labels[i] = NER_TAGS[tag]
        else:
            # Apply AGENT tags (Fallback)
            if agent and str(agent).lower() not in ("null", "none", ""):
                agent_start = text.lower().find(str(agent).lower())
                if agent_start >= 0:
                    agent_end = agent_start + len(str(agent))
                    first = True
                    for i, (s, e) in enumerate(offsets):
                        if s is None or e is None or s == e:
                            continue
                        if e <= agent_start or s >= agent_end:
                            continue
                        if labels[i] == -100:
                            continue
                        labels[i] = NER_TAGS["B-AGENT"] if first else NER_TAGS["I-AGENT"]
                        first = False

            # Apply ACTION tags (Fallback)
            if action and str(action).lower() not in ("null", "none", ""):
                action_start = text.lower().find(str(action).lower())
                if action_start >= 0:
                    action_end = action_start + len(str(action))
                    first = True
                    for i, (s, e) in enumerate(offsets):
                        if s is None or e is None or s == e:
                            continue
                        if e <= action_start or s >= action_end:
                            continue
                        if labels[i] == -100:
                            continue
                        # Don't overwrite AGENT tags
                        if labels[i] in (NER_TAGS["B-AGENT"], NER_TAGS["I-AGENT"]):
                            continue
                        labels[i] = NER_TAGS["B-ACTION"] if first else NER_TAGS["I-ACTION"]
                        first = False

        all_input_ids.append(input_ids)
        all_attention_masks.append(attention_mask)
        all_labels.append(labels)

    return Dataset.from_dict({
        "input_ids": all_input_ids,
        "attention_mask": all_attention_masks,
        "labels": all_labels,
    })


print("Preparing NER datasets (tokenizing + BIO alignment)...")
t0 = time.time()

ner_train_ds = prepare_ner_from_text(ner_train)
ner_val_ds = prepare_ner_from_text(ner_val)
ner_test_ds = prepare_ner_from_text(ner_test)

ner_train_ds.set_format("torch")
ner_val_ds.set_format("torch")
ner_test_ds.set_format("torch")

print(f"NER datasets ready in {time.time() - t0:.1f}s")
print(f"  Train: {len(ner_train_ds)}")
print(f"  Val:   {len(ner_val_ds)}")
print(f"  Test:  {len(ner_test_ds)}")

# Verify label distribution
all_tags = []
for example in ner_train_ds:
    for tag in example["labels"].tolist():
        if tag != -100:
            all_tags.append(tag)

tag_dist = Counter(all_tags)
print(f"\nNER tag distribution (train):")
for tag_id, count in sorted(tag_dist.items()):
    tag_name = ID_TO_NER_TAG[tag_id]
    pct = count / len(all_tags) * 100
    print(f"  {tag_name:10s}: {count:8,d} ({pct:.1f}%)")


# =============================================================================
# === CELL 11: NER Model Setup & Training ===
# =============================================================================
# Expected time: ~30-60 min on P100

print(f"Loading {MODEL_NAME} for token classification...")
ner_model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_NER_TAGS,
    ignore_mismatched_sizes=True,
)

# seqeval metrics
try:
    from seqeval.metrics import f1_score as seq_f1, classification_report as seq_report
    SEQEVAL_AVAILABLE = True
except ImportError:
    print("⚠️ seqeval not available. Using simple token-level metrics.")
    SEQEVAL_AVAILABLE = False


def compute_ner_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    # Flatten, ignoring -100
    true_flat = []
    pred_flat = []
    for i in range(len(labels)):
        for j in range(len(labels[i])):
            if labels[i][j] != -100:
                true_flat.append(labels[i][j])
                pred_flat.append(preds[i][j])

    # Token-level metrics
    macro_f1 = f1_score(true_flat, pred_flat, average="macro", zero_division=0)
    micro_f1 = f1_score(true_flat, pred_flat, average="micro", zero_division=0)

    # Entity-level metrics with seqeval (if available)
    if SEQEVAL_AVAILABLE:
        true_tags = [[ID_TO_NER_TAG[t] for t in labels[i] if t != -100] for i in range(len(labels))]
        pred_tags = [[ID_TO_NER_TAG[preds[i][j]] for j in range(len(labels[i])) if labels[i][j] != -100]
                     for i in range(len(labels))]
        entity_f1 = seq_f1(true_tags, pred_tags)
    else:
        entity_f1 = macro_f1

    return {
        "token_macro_f1": macro_f1,
        "token_micro_f1": micro_f1,
        "entity_f1": entity_f1,
    }


NER_OUTPUT_DIR = "/kaggle/working/clauseops-ner-extractor"
if not os.path.exists("/kaggle/working"):
    NER_OUTPUT_DIR = "./clauseops-ner-extractor"

ner_training_args = TrainingArguments(
    output_dir=NER_OUTPUT_DIR,
    num_train_epochs=15,   # NER often needs more epochs
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=4,
    learning_rate=3e-5,   # Slightly higher LR for token classification
    warmup_ratio=0.1,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="entity_f1",
    greater_is_better=True,
    save_total_limit=2,
    logging_steps=50,
    logging_first_step=True,
    report_to="none",
    seed=42,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
)

ner_trainer = Trainer(
    model=ner_model,
    args=ner_training_args,
    train_dataset=ner_train_ds,
    eval_dataset=ner_val_ds,
    compute_metrics=compute_ner_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

print("🚀 Training Model B: Agent + Action NER...")
print(f"   Train size: {len(ner_train_ds)}")
print(f"   Epochs: {ner_training_args.num_train_epochs}")
print()

ner_train_result = ner_trainer.train()

print("\n" + "=" * 50)
print("MODEL B TRAINING COMPLETE")
print("=" * 50)
ner_metrics = ner_train_result.metrics
print(f"Training loss: {ner_metrics.get('train_loss', 'N/A'):.4f}")
print(f"Training time: {ner_metrics.get('train_runtime', 0) / 60:.1f} minutes")


# =============================================================================
# === CELL 12: Evaluate Model B ===
# =============================================================================

print("Evaluating Model B on test set...")
ner_eval = ner_trainer.evaluate(ner_test_ds)

print(f"\n{'=' * 50}")
print("MODEL B — TEST SET RESULTS")
print(f"{'=' * 50}")
print(f"Token Macro-F1:  {ner_eval['eval_token_macro_f1']:.4f}")
print(f"Token Micro-F1:  {ner_eval['eval_token_micro_f1']:.4f}")
print(f"Entity F1:       {ner_eval['eval_entity_f1']:.4f}")

# Detailed predictions
ner_preds = ner_trainer.predict(ner_test_ds)
pred_tags_flat = []
true_tags_flat = []
for i in range(len(ner_preds.label_ids)):
    for j in range(len(ner_preds.label_ids[i])):
        if ner_preds.label_ids[i][j] != -100:
            true_tags_flat.append(ner_preds.label_ids[i][j])
            pred_tags_flat.append(np.argmax(ner_preds.predictions[i][j]))

ner_tag_names = [ID_TO_NER_TAG[i] for i in range(NUM_NER_TAGS)]
print(f"\n{'=' * 60}")
print("DETAILED NER CLASSIFICATION REPORT (token-level)")
print(f"{'=' * 60}")
print(classification_report(
    true_tags_flat, pred_tags_flat,
    target_names=ner_tag_names,
    digits=3,
    zero_division=0,
))


# =============================================================================
# === CELL 13: Save Model B ===
# =============================================================================

ner_save_path = os.path.join(NER_OUTPUT_DIR, "final")
print(f"Saving Model B to: {ner_save_path}")

ner_trainer.save_model(ner_save_path)
tokenizer.save_pretrained(ner_save_path)

ner_metadata = {
    "model_name": MODEL_NAME,
    "task": "ner_agent_action",
    "num_labels": NUM_NER_TAGS,
    "label_map": NER_TAGS,
    "id_to_label": {str(v): k for k, v in NER_TAGS.items()},
    "eval_entity_f1": ner_eval.get("eval_entity_f1", 0),
    "eval_token_macro_f1": ner_eval.get("eval_token_macro_f1", 0),
    "training_date": time.strftime("%Y-%m-%d %H:%M"),
}

with open(os.path.join(ner_save_path, "ner_metadata.json"), "w") as f:
    json.dump(ner_metadata, f, indent=2)

print("✅ Model B saved!")
for fname in sorted(os.listdir(ner_save_path)):
    fsize = os.path.getsize(os.path.join(ner_save_path, fname))
    print(f"   {fname}: {fsize / 1e6:.1f} MB")


# =============================================================================
# === CELL 14: Quick Inference Test (Both Models) ===
# =============================================================================

print("=" * 70)
print("INFERENCE TEST — BOTH MODELS")
print("=" * 70)

test_clauses = [
    "ESSI will feature the following disclaimer in close proximity to said endorsement.",
    "PrimeCall shall negotiate and contract on behalf of DeltaThree for the printing of the Calling Cards.",
    "All matters relating to this Agreement will be treated by the Members as confidential.",
    "This Agreement shall be governed by the laws of the State of New York.",
    "Affiliate agrees to place Chase's links provided by Linkshare Network on its website.",
    "Either party may terminate this Agreement upon thirty days written notice.",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Model A: Modality
mod_model.eval()
mod_model.to(device)

# Model B: NER
ner_model.eval()
ner_model.to(device)

for clause in test_clauses:
    inputs = tokenizer(clause, return_tensors="pt", truncation=True,
                        max_length=MAX_LENGTH, padding=True).to(device)

    # Modality prediction
    with torch.no_grad():
        mod_logits = mod_model(**inputs).logits
    mod_probs = torch.softmax(mod_logits, dim=-1)[0]
    mod_pred_id = mod_probs.argmax().item()
    modality = ID_TO_MODALITY[mod_pred_id]
    mod_conf = mod_probs[mod_pred_id].item()

    # NER prediction
    with torch.no_grad():
        ner_logits = ner_model(**inputs).logits
    ner_preds = ner_logits.argmax(dim=-1)[0]
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    # Extract agent and action spans
    agent_tokens = []
    action_tokens = []
    for tok, tag_id in zip(tokens, ner_preds.tolist()):
        tag = ID_TO_NER_TAG.get(tag_id, "O")
        if tag in ("B-AGENT", "I-AGENT"):
            agent_tokens.append(tok)
        elif tag in ("B-ACTION", "I-ACTION"):
            action_tokens.append(tok)

    agent = tokenizer.convert_tokens_to_string(agent_tokens) if agent_tokens else "N/A"
    action = tokenizer.convert_tokens_to_string(action_tokens) if action_tokens else "N/A"

    print(f"\nClause: {clause[:80]}...")
    print(f"  Modality: {modality} ({mod_conf:.2f})")
    print(f"  Agent:    {agent}")
    print(f"  Action:   {action}")

print("\n✅ Inference test complete!")
print("\n" + "=" * 70)
print("NEXT STEPS:")
print("  1. Download both model folders from /kaggle/working/")
print("  2. Place them in your ClauseOps project:")
print("     - clauseops/obligation_detection/models/modality-classifier/")
print("     - clauseops/obligation_detection/models/ner-extractor/")
print("  3. Run the integration script to wire them into the pipeline")
print("=" * 70)


NER Tags: {'O': 0, 'B-AGENT': 1, 'I-AGENT': 2, 'B-ACTION': 3, 'I-ACTION': 4}
Number of NER tags: 5
Preparing NER datasets (tokenizing + BIO alignment)...
NER datasets ready in 3.1s
  Train: 1498
  Val:   249
  Test:  251

NER tag distribution (train):
  O         :  143,117 (86.1%)
  B-AGENT   :      749 (0.5%)
  I-AGENT   :    1,410 (0.8%)
  B-ACTION  :      749 (0.5%)
  I-ACTION  :   20,222 (12.2%)
Loading nlpaueb/legal-bert-base-uncased for token classification...


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: nlpaueb/legal-bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored w

🚀 Training Model B: Agent + Action NER...
   Train size: 1498
   Epochs: 15



/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Token Macro F1,Token Micro F1,Entity F1
1,12.328046,1.392801,0.183443,0.846151,0.000000
2,12.328046,0.886117,0.184695,0.856784,0.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


In [ ]:
import shutil

# Create separate zip files
shutil.make_archive(
    '/kaggle/working/clauseops-modality-classifier',
    'zip',
    '/kaggle/working/clauseops-modality-classifier'
)

shutil.make_archive(
    '/kaggle/working/clauseops-ner-extractor',
    'zip',
    '/kaggle/working/clauseops-ner-extractor'
)

print("ZIP files created:")
print("/kaggle/working/clauseops-modality-classifier.zip")
print("/kaggle/working/clauseops-ner-extractor.zip")